In [1]:
from pathlib import Path
import subprocess
import re

In [2]:
#we make the pddl text of predicates
domain_text = """(define (domain urbanbot_delivery)
  (:requirements :strips :typing)
  
  (:types 
    ubicacion 
    robot 
    paquete
  )
  
  (:predicates
    (en_robot ?r - robot ?loc - ubicacion)       ; Ubicación actual del robot
    (en_paquete ?p - paquete ?loc - ubicacion)   ; Ubicación del paquete 
    (bateria_cargada ?r - robot)                 ; El robot tiene batería suficiente
    (cargador_en ?loc - ubicacion)               ; Hay estación de carga en esa ubicación 
    (lleva_paquete ?r - robot ?p - paquete)      ; El robot transporta el paquete 
    (entregado ?p - paquete ?loc - ubicacion)     ; El paquete ha sido entregado 
    (adyacente ?loc1 - ubicacion ?loc2 - ubicacion) ; Conexión directa entre dos ubicaciones
  )

  ;; Acción 1: Mover el robot entre ubicaciones adyacentes
  (:action mover
    :parameters (?r - robot ?desde - ubicacion ?hasta - ubicacion)
    :precondition (and 
      (en_robot ?r ?desde) 
      (bateria_cargada ?r) 
      (adyacente ?desde ?hasta)
    )
    :effect (and 
      (en_robot ?r ?hasta) 
      (not (en_robot ?r ?desde))
    )
  )

  ;; Acción 2: Recoger un paquete en la ubicación actual
  (:action recoger_paquete
    :parameters (?r - robot ?p - paquete ?loc - ubicacion)
    :precondition (and 
      (en_robot ?r ?loc) 
      (en_paquete ?p ?loc)
    )
    :effect (and 
      (lleva_paquete ?r ?p) 
      (not (en_paquete ?p ?loc))
    )
  )

  ;; Acción 3: Entregar el paquete en la ubicación de destino
  (:action entregar_paquete
    :parameters (?r - robot ?p - paquete ?loc - ubicacion)
    :precondition (and 
      (en_robot ?r ?loc) 
      (lleva_paquete ?r ?p)
    )
    :effect (and 
      (entregado ?p ?loc) 
      (not (lleva_paquete ?r ?p))
    )
  )

  ;; Acción 4: Recargar la batería del robot en un punto de carga
  (:action recargar
    :parameters (?r - robot ?loc - ubicacion)
    :precondition (and 
      (en_robot ?r ?loc) 
      (cargador_en ?loc)
    )
    :effect (and 
      (bateria_cargada ?r)
    )
  )
)
"""

out_dir = Path('pddl_urbanbot')
out_dir.mkdir(exist_ok=True)
domain_path = out_dir / 'domain_urbanbot.pddl'
domain_path.write_text(domain_text, encoding='utf-8')

print("PDDL domain file successfully generated at:")
print(domain_path.resolve())

PDDL domain file successfully generated at:
C:\Users\Usuario\PORTFOLIO\Portfolio in English\pddl_urbanbot\domain_urbanbot.pddl


In [3]:
#we create the problem in pddl
problem_text = """(define (problem urbanbot_mission)
  (:domain urbanbot_delivery)
  
  (:objects
    almacen puntoA puntoB - ubicacion
    bot - robot
    paquete1 - paquete
  )
  
  (:init
    ;; Estado de ubicación inicial del robot y el paquete
    (en_robot bot puntoA)
    (en_paquete paquete1 almacen) 
    
    ;; Infraestructura disponible (Estación en Punto A)
    (cargador_en puntoA) 
    
    ;; Nota: No incluimos (bateria_cargada bot) para simular la batería baja
    
    ;; Declaración de la topología lineal de la red urbana
    (adyacente almacen puntoA) 
    (adyacente puntoA almacen) 
    (adyacente puntoA puntoB) 
    (adyacente puntoB puntoA) 
  )
  
  (:goal
    (and 
      (entregado paquete1 puntoB) 
    )
  )
)
"""

problem_path = out_dir / 'problem_urbanbot.pddl'
problem_path.write_text(problem_text, encoding='utf-8')

print("PDDL problem file successfully generated at:")
print(problem_path.resolve())

PDDL problem file successfully generated at:
C:\Users\Usuario\PORTFOLIO\Portfolio in English\pddl_urbanbot\problem_urbanbot.pddl


In [4]:
fd_path = Path(r"C:\Users\Usuario\AppData\Local\Programs\Python\Python311\Lib\site-packages\up_fast_downward\downward\fast-downward.py")
domain_path = Path("pddl_urbanbot/domain_urbanbot.pddl")
problem_path = Path("pddl_urbanbot/problem_urbanbot.pddl")
plan_path = Path("sas_plan")

print("Domain exists:", domain_path.exists(), domain_path.resolve())
print("Problem exists:", problem_path.exists(), problem_path.resolve())

cmd = [
    "python", str(fd_path),
    str(domain_path),
    str(problem_path),
    "--search", "astar(lmcut())"
]

result = subprocess.run(cmd, capture_output=True, text=True)

if plan_path.exists():
    print("
=== PLAN FOUND ===")
    print(plan_path.read_text(encoding="utf-8"))
else:
    print("\nNo sas_plan was generated.")

SyntaxError: unterminated string literal (detected at line 19) (864407989.py, line 19)

In [ ]:
plan_text = plan_path.read_text(encoding="utf-8")
#we parse the plan
acciones_plan = []
for line in plan_text.splitlines():
    line = line.strip()
    if not line or line.startswith(";"):
        continue
    m = re.search(r"\((.*?)\)", line)
    if m:
        tokens = m.group(1).lower().split()
        nombre = tokens[0]
        params = tuple(tokens[1:])
        acciones_plan.append((nombre, params))

print("PARSED ACTIONS")
for i, (accion, params) in enumerate(acciones_plan, 1):
    print(f"{i}. {accion}{params}")

#we define the initial state
estado_actual = {
    "en_robot(bot,puntoa)",
    "en_paquete(paquete1,almacen)",
    "cargador_en(puntoa)",
    "adyacente(almacen,puntoa)",
    "adyacente(puntoa,almacen)",
    "adyacente(puntoa,puntob)",
    "adyacente(puntob,puntoa)"
}

objetivo = "entregado(paquete1,puntob)"

def hechos_fluidos(estado):
    return sorted([p for p in estado if not p.startswith("adyacente")])

def verificar_precondiciones(accion, params, state):
    if accion == "recargar":
        r, loc = params
        return [
            f"en_robot({r},{loc})" in state,
            f"cargador_en({loc})" in state
        ]
    elif accion == "mover":
        r, desde, hasta = params
        return [
            f"en_robot({r},{desde})" in state,
            f"bateria_cargada({r})" in state,
            f"adyacente({desde},{hasta})" in state
        ]
    elif accion == "recoger_paquete":
        r, p, loc = params
        return [
            f"en_robot({r},{loc})" in state,
            f"en_paquete({p},{loc})" in state
        ]
    elif accion == "entregar_paquete":
        r, p, loc = params
        return [
            f"en_robot({r},{loc})" in state,
            f"lleva_paquete({r},{p})" in state
        ]
    else:
        raise ValueError(f"Unknown action: {accion}")

def apply_action(action_name, params, state):
    new_state = set(state)

    if action_name == "recargar":
        r, loc = params
        if f"en_robot({r},{loc})" in state and f"cargador_en({loc})" in state:
            new_state.add(f"bateria_cargada({r})")
            return new_state
        raise ValueError(f"Precondition failure in recargar({r},{loc})")

    elif action_name == "mover":
        r, desde, hasta = params
        if (
            f"en_robot({r},{desde})" in state and
            f"bateria_cargada({r})" in state and
            f"adyacente({desde},{hasta})" in state
        ):
            new_state.remove(f"en_robot({r},{desde})")
            new_state.add(f"en_robot({r},{hasta})")
            return new_state
        raise ValueError(f"Precondition failure in mover({r},{desde},{hasta})")

    elif action_name == "recoger_paquete":
        r, p, loc = params
        if f"en_robot({r},{loc})" in state and f"en_paquete({p},{loc})" in state:
            new_state.remove(f"en_paquete({p},{loc})")
            new_state.add(f"lleva_paquete({r},{p})")
            return new_state
        raise ValueError(f"Precondition failure in recoger_paquete({r},{p},{loc})")

    elif action_name == "entregar_paquete":
        r, p, loc = params
        if f"en_robot({r},{loc})" in state and f"lleva_paquete({r},{p})" in state:
            new_state.remove(f"lleva_paquete({r},{p})")
            new_state.add(f"entregado({p},{loc})")
            return new_state
        raise ValueError(f"Precondition failure in entregar_paquete({r},{p},{loc})")

    else:
        raise ValueError(f"Unknown action: {action_name}")

#we verify step by step that the preconditions of each action are met
print("PRECONDITIONS VERIFICATION")
print("Initial state:", hechos_fluidos(estado_actual), "\n")

for paso, (accion, params) in enumerate(acciones_plan, 1):
    checks = verificar_precondiciones(accion, params, estado_actual)
    print(f"Step {paso}: {accion}{params}")
    print("Satisfied preconditions:", all(checks), checks)

    if not all(checks):
        raise ValueError(f"Preconditions are not met at step {paso}: {accion}{params}")

    estado_actual = apply_action(accion, params, estado_actual)
    print("Resulting state:", hechos_fluidos(estado_actual), "\n")

print("GOAL SATISFACTION VERIFICATION")
print("Goal sought:", objetivo)
print("Goal reached?", objetivo in estado_actual)

if objetivo in estado_actual:
    print("COMPLETE VERIFICATION: the final state satisfies the goal.")
else:
    print("ERROR: the plan does not reach the goal.")

### **1. How many actions does the minimum plan have? Is it optimal or simply sufficient?**
The minimum plan consists of 6 actions. This plan is optimal, since given the linear topology of the environment (Warehouse – Point A – Point B) and the restriction that there is no direct connection between the Warehouse and Point B, there is no shorter sequence of steps to meet the objective.

### **2. What would happen if the charging station were at the Warehouse instead of Point A?**
In this scenario, the problem would become unsolvable, because the robot starts at Point A with "low battery" and the `mover` action obligatorily requires the `bateria_cargada` predicate. If the charger were at the Warehouse, the robot would not have enough energy to perform the initial movement towards said location, getting stuck in the initial state.

### **3. What PDDL extension would you use if the battery were numeric and consumed with each movement?**
The PDDL 2.1 extension of the language introduced numeric fluents (quantitative resources) and durative actions, which allows modeling variables that change value arithmetically, such as the charge level of a battery that decreases proportionally to use, which is our case.

### **4. What advantage would a heuristic planner have over BFS in a larger domain?**
The main advantage is exploration efficiency: while BFS performs an uninformed search that expands all states level by level (suffering a combinatorial explosion), a heuristic planner uses a function (such as FF or HSP) to estimate the distance to the goal, which allows directing the search towards the most promising branches and drastically reducing the number of expanded nodes and resolution time.

### **5. In what scenario would it make sense to use HTN, GraphPlan, or SATPlan instead of STRIPS?**
*   HTN is ideal when the problem can be decomposed into a hierarchy of tasks and subtasks, allowing the encoding of expert knowledge about logical procedures.
*   GraphPlan makes sense in domains with high concurrency since it uses planning graphs and mutual exclusions to identify actions that can be executed in parallel without interfering with each other.
*   SATPlan is useful when it is preferred to translate the problem to a logical satisfiability formula**, taking advantage of the power of modern SAT-solvers to find solutions in highly structured domains.